# 01 — Phát triển Hybrid Baseline Classifier (PyTorch)

Notebook này train 1 mạng CNN 1D nhỏ (`BaselineClassifier` trong
`src/hybrid_baseline.py`) để nhận diện phổ đang bị trội bởi baseline dạng
**đa thức bậc 2**, **Gaussian**, **cả hai**, hay **không đáng kể** — dùng
làm bước "định hướng" trước khi tinh chỉnh bằng airPLS
(`hybrid_baseline_correction`).

**Khác với dự án tham khảo (anh khóa trên):**
- Chuyển từ Keras/TensorFlow sang **PyTorch** (khớp `models/baseline_classifier.pt`).
- **Bỏ hoàn toàn phần mã hoá GADF/ảnh 2D** — chỉ làm việc trên phổ 1D.
- Nhãn train được sinh ra **có kiểm soát** (biết chính xác baseline nào đã
  thêm vào), thay vì dùng file `labels_noise_pure_182.npy` không rõ nguồn
  gốc như bản gốc.
- Tinh chỉnh baseline dùng **airPLS sẵn có** trong `raman_processing.py`
  (cùng `lam` với `00_parameter_selection.ipynb`), không viết thêm 1 bản
  airPLS thứ hai.

**Chạy notebook này TRƯỚC `02_clean_augment_export.ipynb`** — output là
`models/baseline_classifier.pt`, dùng làm tuỳ chọn baseline correction thay
thế (bên cạnh airPLS thuần) cho các notebook sau.

In [ ]:
import sys, os, json
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

from raman_processing import baseline_correction
from hybrid_baseline import (
    poly_baseline, gaussian_baseline, pg_baseline,
    BaselineClassifier, save_baseline_classifier, normalize_minmax,
    hybrid_baseline_correction,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

os.makedirs('../models', exist_ok=True)

with open('../data/processed/chosen_params.json') as f:
    params = json.load(f)
AIRPLS_LAM = params['airpls_lam']
print('AIRPLS_LAM (từ 00_parameter_selection.ipynb):', AIRPLS_LAM)

## 1. Đọc phổ gốc, tạo tập phổ "sạch tham chiếu"

Dùng airPLS sẵn có để lấy baseline THẬT ĐÃ BIẾT LÀ GẦN 0 (đã hiệu chỉnh),
sau đó chuẩn hoá về [0,1] -- đây là "nguyên liệu" để cộng thêm baseline giả
lập (biết trước tham số) và tạo nhãn huấn luyện chính xác.

In [ ]:
df_raw = pd.read_excel('../data/raw/Ethanol_Methanol.xlsx')
x_full = df_raw['Raman Shift (cm-1)'].values
mask = ~np.isnan(x_full)
x = x_full[mask]
df = df_raw.loc[mask].reset_index(drop=True)
sample_cols = [c for c in df.columns if c != 'Raman Shift (cm-1)']

SPECTRUM_LENGTH = len(x)
print(f'{len(sample_cols)} mẫu, {SPECTRUM_LENGTH} điểm/phổ')

clean_refs = []
for col in sample_cols:
    corrected = baseline_correction(df[col].values.astype(float), method='airpls', lam=AIRPLS_LAM)
    clean_refs.append(normalize_minmax(corrected))
clean_refs = np.array(clean_refs, dtype=np.float32)
print('clean_refs shape:', clean_refs.shape)

## 2. Sinh dữ liệu train có nhãn CHÍNH XÁC cho classifier

Với mỗi phổ sạch tham chiếu, sinh nhiều bản sao có cộng thêm baseline giả
lập theo 4 tình huống: **không có / chỉ poly / chỉ gaussian / cả hai** --
nhãn `[poly_present, gaussian_present]` biết chính xác vì ta tự thêm vào.
Tham số baseline lấy khoảng giá trị tương tự code tham khảo, điều chỉnh lại
biên độ cho phù hợp phổ đã chuẩn hoá [0,1].

In [ ]:
N_PER_SPECTRUM = 40  # số bản sao sinh ra cho mỗi phổ sạch tham chiếu
x_range = np.linspace(0, SPECTRUM_LENGTH, SPECTRUM_LENGTH)

def make_example(clean_spec, rng):
    kind = rng.choice(['none', 'poly', 'gaussian', 'both'], p=[0.25, 0.25, 0.25, 0.25])
    s = clean_spec.copy()
    s = s + rng.normal(0, 0.02, size=SPECTRUM_LENGTH)  # nhiễu nền nhỏ, luôn có

    label = np.array([0.0, 0.0], dtype=np.float32)

    if kind in ('poly', 'both'):
        s = s + poly_baseline(x_range, p=rng.uniform(1.5, 3.0),
                               intensity=rng.uniform(0.3, 0.9), b=rng.uniform(-0.1, 0.1))
        label[0] = 1.0
    if kind in ('gaussian', 'both'):
        s = s + gaussian_baseline(x_range, mean=rng.uniform(0, SPECTRUM_LENGTH),
                                   sd=rng.uniform(60, 120), intensity=rng.uniform(0.3, 0.9),
                                   b=rng.uniform(-0.1, 0.1))
        label[1] = 1.0

    return s.astype(np.float32), label


rng = np.random.default_rng(SEED)
X_all, y_all = [], []
for spec in clean_refs:
    for _ in range(N_PER_SPECTRUM):
        s, lbl = make_example(spec, rng)
        X_all.append(s)
        y_all.append(lbl)

X_all = np.array(X_all, dtype=np.float32)
y_all = np.array(y_all, dtype=np.float32)
print('X_all:', X_all.shape, ' y_all:', y_all.shape)
print('Phân bố nhãn (poly, gaussian):')
print('  chỉ poly:    ', int(((y_all[:,0]==1) & (y_all[:,1]==0)).sum()))
print('  chỉ gaussian:', int(((y_all[:,0]==0) & (y_all[:,1]==1)).sum()))
print('  cả hai:      ', int(((y_all[:,0]==1) & (y_all[:,1]==1)).sum()))
print('  không có:    ', int(((y_all[:,0]==0) & (y_all[:,1]==0)).sum()))

X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=SEED)
print('\nTrain:', X_train.shape, ' Val:', X_val.shape)

## 3. Train BaselineClassifier (PyTorch)

In [ ]:
model = BaselineClassifier(input_length=SPECTRUM_LENGTH).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()  # khớp đúng với output logit thô + nhãn multi-label

EPOCHS = 40
BATCH_SIZE = 32

X_train_t = torch.tensor(X_train, device=DEVICE)
y_train_t = torch.tensor(y_train, device=DEVICE)
X_val_t = torch.tensor(X_val, device=DEVICE)
y_val_t = torch.tensor(y_val, device=DEVICE)

n_train = len(X_train_t)
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(n_train)
    total_loss = 0.0
    for i in range(0, n_train, BATCH_SIZE):
        idx = perm[i:i + BATCH_SIZE]
        xb, yb = X_train_t[idx], y_train_t[idx]
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(idx)
    train_loss = total_loss / n_train

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_loss = criterion(val_logits, y_val_t).item()
        val_pred = (torch.sigmoid(val_logits) >= 0.5).float()
        val_acc = (val_pred == y_val_t).float().mean().item()

    history.append((epoch, train_loss, val_loss, val_acc))
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.3f}')

print('\nHoàn tất train. Val accuracy cuối:', history[-1][3])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
hist_arr = np.array(history)
axes[0].plot(hist_arr[:,0], hist_arr[:,1], label='train_loss')
axes[0].plot(hist_arr[:,0], hist_arr[:,2], label='val_loss')
axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].set_title('Loss')
axes[1].plot(hist_arr[:,0], hist_arr[:,3])
axes[1].set_xlabel('epoch'); axes[1].set_title('Val accuracy (per-label, ngưỡng 0.5)')
plt.tight_layout()
plt.savefig('../results/figures/hybrid_baseline_training.png', dpi=120)
plt.show()

## 4. Lưu model -> `models/baseline_classifier.pt`

File này được `hybrid_baseline.load_baseline_classifier()` đọc lại ở các
notebook sau (vd nếu bạn muốn thử `hybrid_baseline_correction` thay cho
airPLS thuần trong `02_clean_augment_export.ipynb`).

In [ ]:
save_baseline_classifier(model, '../models/baseline_classifier.pt')
print('Đã lưu ../models/baseline_classifier.pt')

## 5. Demo: so sánh airPLS thuần vs hybrid (DL-guided + airPLS)

Thử trên vài phổ THẬT (không phải phổ tổng hợp) để xem hybrid có khác biệt
rõ rệt so với airPLS thuần không. Đây là bước tham khảo trực quan, KHÔNG
phải đánh giá định lượng (muốn định lượng cần so sánh SNR/peak fitting như
`00_parameter_selection.ipynb` đã làm cho airPLS).

In [ ]:
model.eval()
demo_cols = sample_cols[:3]

fig, axes = plt.subplots(len(demo_cols), 1, figsize=(9, 3 * len(demo_cols)))
if len(demo_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, demo_cols):
    raw = df[col].values.astype(float)
    raw_n = normalize_minmax(raw)

    airpls_only = normalize_minmax(baseline_correction(raw, method='airpls', lam=AIRPLS_LAM))
    hybrid = hybrid_baseline_correction(raw_n, model, device=DEVICE, airpls_lam=AIRPLS_LAM)

    ax.plot(x, raw_n, label='raw (chuẩn hoá)', alpha=0.4, color='gray')
    ax.plot(x, airpls_only, label='airPLS thuần', alpha=0.8)
    ax.plot(x, hybrid, label='hybrid (DL-guided + airPLS)', alpha=0.8)
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../results/figures/hybrid_vs_airpls_demo.png', dpi=120)
plt.show()

## Tổng kết

- Model đã lưu: `models/baseline_classifier.pt`
- Dùng `hybrid_baseline.hybrid_baseline_correction(spectrum, model, device=DEVICE, airpls_lam=AIRPLS_LAM)`
  ở bất kỳ notebook nào cần baseline correction, thay cho
  `baseline_correction(..., method='airpls')` thuần nếu muốn thử nghiệm.
- **Lưu ý:** đây là bước DL-guided THÊM VÀO trước airPLS, không thay thế
  hoàn toàn airPLS -- airPLS vẫn luôn chạy ở bước tinh chỉnh cuối, nên
  hybrid không thể kém ổn định hơn airPLS thuần theo thiết kế (fallback tự
  động nếu bước DL-guided lỗi).